In [22]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/btc_features.csv", parse_dates=["time"])

# Target: did the NEXT hour close up? (1 = yes, 0 = no)
df["next_ret"] = df["return"].shift(-1)
df["next_up"] = (df["next_ret"] > 0).astype(int)
df = df.iloc[:-1]   # last row has no next hour yet

# Time-based split: first 80% to train, last 20% to test
split = int(len(df) * 0.8)
train = df.iloc[:split]
test = df.iloc[split:]

print("Train:", train.shape, train["time"].min(), "->", train["time"].max())
print("Test: ", test.shape, test["time"].min(), "->", test["time"].max())

Train: (3438, 10) 2026-05-09 14:00:00 -> 2026-09-29 19:00:00
Test:  (860, 10) 2026-09-29 20:00:00 -> 2026-11-04 15:00:00


In [23]:
print("Share of up hours in train:", round(train["next_up"].mean() * 100, 1), "%")
print("Share of up hours in test: ", round(test["next_up"].mean() * 100, 1), "%")

Share of up hours in train: 49.9 %
Share of up hours in test:  49.3 %


In [24]:
def accuracy(pred, actual):
    return round((pred == actual).mean() * 100, 1)

y_test = test["next_up"]

# Baseline 1: always predict whichever class was more common in TRAIN
majority = int(train["next_up"].mean() > 0.5)
pred1 = pd.Series(majority, index=test.index)
print("Always", "up" if majority else "down", ":", accuracy(pred1, y_test), "%")

# Baseline 2: next hour goes the same direction as this hour
pred2 = (test["return"] > 0).astype(int)
print("Same as last hour:    ", accuracy(pred2, y_test), "%")

# Baseline 3: next hour goes the opposite direction
pred3 = 1 - pred2
print("Opposite of last hour:", accuracy(pred3, y_test), "%")

Always down : 50.7 %
Same as last hour:     48.6 %
Opposite of last hour: 51.4 %


In [25]:
for name, part in [("train", train), ("test", test)]:
    same = ((part["return"] > 0).astype(int) == part["next_up"]).mean()
    print(name, "| same-as-last accuracy:", round(same * 100, 1), "%",
          "| opposite:", round((1 - same) * 100, 1), "%")

train | same-as-last accuracy: 47.3 % | opposite: 52.7 %
test | same-as-last accuracy: 48.6 % | opposite: 51.4 %


In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

feature_cols = ["return", "ret_5", "vol_20", "rsi_14", "vol_ratio", "dist_ma"]

X_train, y_train = train[feature_cols], train["next_up"]
X_test, y_test = test[feature_cols], test["next_up"]

model = make_pipeline(StandardScaler(), LogisticRegression())
model.fit(X_train, y_train)

pred_train = model.predict(X_train)
pred_test = model.predict(X_test)

print("Train accuracy:", round((pred_train == y_train).mean() * 100, 1), "%")
print("Test accuracy: ", round((pred_test == y_test).mean() * 100, 1), "%")
print("Share of 'up' predictions in test:", round(pred_test.mean() * 100, 1), "%")

Train accuracy: 52.3 %
Test accuracy:  51.0 %
Share of 'up' predictions in test: 41.7 %


In [27]:
coefs = pd.Series(model[-1].coef_[0], index=feature_cols).round(3)
print(coefs.sort_values())

ret_5       -0.114
rsi_14      -0.070
return      -0.068
vol_20      -0.010
vol_ratio    0.053
dist_ma      0.090
dtype: float64


In [28]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)
xgb.fit(X_train, y_train)

pred_train_x = xgb.predict(X_train)
pred_test_x = xgb.predict(X_test)

print("Train accuracy:", accuracy(pred_train_x, y_train), "%")
print("Test accuracy: ", accuracy(pred_test_x, y_test), "%")
print("Share of 'up' predictions in test:", round(pred_test_x.mean() * 100, 1), "%")

Train accuracy: 69.1 %
Test accuracy:  51.0 %
Share of 'up' predictions in test: 51.7 %


In [29]:
imp = pd.Series(xgb.feature_importances_, index=feature_cols).round(3)
print(imp.sort_values(ascending=False))

return       0.176
vol_ratio    0.175
ret_5        0.170
vol_20       0.161
dist_ma      0.160
rsi_14       0.158
dtype: float32


In [30]:
results = pd.DataFrame({
    "model": ["Always down", "Same as last hour", "Opposite of last hour",
              "Logistic Regression", "XGBoost"],
    "test_accuracy_%": [
        accuracy(pred1, y_test), accuracy(pred2, y_test), accuracy(pred3, y_test),
        accuracy(pred_test, y_test), accuracy(pred_test_x, y_test),
    ],
})
print(results.to_string(index=False))

                model  test_accuracy_%
          Always down             50.7
    Same as last hour             48.6
Opposite of last hour             51.4
  Logistic Regression             51.0
              XGBoost             51.0


In [31]:
FEE = 0.001   # 0.1% per trade (assumption)

def backtest(pred, data, fee=FEE):
    pos = pd.Series(np.asarray(pred), index=data.index)   # 1 = holding BTC, 0 = cash
    prev = pos.shift(1).fillna(0)
    trades = (pos - prev).abs()                            # 1 whenever we switch
    gross = pos * data["next_ret"]                         # return before fees
    net = gross - trades * fee                             # return after fees
    return gross, net, trades

models = {
    "Opposite of last hour": pred3,
    "Logistic Regression": pred_test,
    "XGBoost": pred_test_x,
}

rows = []
for name, p in models.items():
    gross, net, trades = backtest(p, test)
    rows.append({
        "strategy": name,
        "gross_%": round(((1 + gross).prod() - 1) * 100, 1),
        "net_%": round(((1 + net).prod() - 1) * 100, 1),
        "trades": int(trades.sum()),
    })

hold = (1 + test["next_ret"]).prod() - 1
rows.append({"strategy": "Buy and hold", "gross_%": round(hold * 100, 1),
             "net_%": round(hold * 100, 1), "trades": 1})

print(pd.DataFrame(rows).to_string(index=False))

             strategy  gross_%  net_%  trades
Opposite of last hour     -5.4  -39.2     442
  Logistic Regression     -3.1  -26.2     272
              XGBoost      0.3  -28.8     342
         Buy and hold     -3.7   -3.7       1


In [32]:
proba = {
    "Logistic Regression": model.predict_proba(X_test)[:, 1],
    "XGBoost": xgb.predict_proba(X_test)[:, 1],
}

for name, p in proba.items():
    s = pd.Series(p)
    print(name, "| min", round(s.min(), 3),
          "| 90th percentile", round(s.quantile(0.9), 3),
          "| max", round(s.max(), 3))

Logistic Regression | min 0.405 | 90th percentile 0.54 | max 0.727
XGBoost | min 0.159 | 90th percentile 0.592 | max 0.802


In [33]:
rows = []
for name, p in proba.items():
    for thr in [0.50, 0.52, 0.55, 0.60]:
        pred = (p > thr).astype(int)
        gross, net, trades = backtest(pred, test)
        hit = y_test.values[pred == 1].mean() * 100 if pred.sum() > 0 else float("nan")
        rows.append({
            "model": name,
            "threshold": thr,
            "hours_in_market": int(pred.sum()),
            "hit_rate_%": round(hit, 1),
            "trades": int(trades.sum()),
            "gross_%": round(((1 + gross).prod() - 1) * 100, 1),
            "net_%": round(((1 + net).prod() - 1) * 100, 1),
        })

print(pd.DataFrame(rows).to_string(index=False))

              model  threshold  hours_in_market  hit_rate_%  trades  gross_%  net_%
Logistic Regression       0.50              359        50.4     272     -3.1  -26.2
Logistic Regression       0.52              172        47.7     150     -4.7  -18.0
Logistic Regression       0.55               56        51.8      74      2.1   -5.2
Logistic Regression       0.60               11        36.4      18     -1.9   -3.6
            XGBoost       0.50              445        50.3     342      0.3  -28.8
            XGBoost       0.52              329        50.5     334     -0.3  -28.6
            XGBoost       0.55              213        51.2     248      2.6  -20.0
            XGBoost       0.60               72        54.2     100      1.4   -8.3


In [34]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=6, test_size=600)

for i, (tr_idx, te_idx) in enumerate(tscv.split(df)):
    print(f"Fold {i+1}: train {len(tr_idx)} rows "
          f"({df['time'].iloc[tr_idx[0]].date()} -> {df['time'].iloc[tr_idx[-1]].date()}) | "
          f"test {len(te_idx)} rows "
          f"({df['time'].iloc[te_idx[0]].date()} -> {df['time'].iloc[te_idx[-1]].date()})")

Fold 1: train 698 rows (2026-05-09 -> 2026-06-07) | test 600 rows (2026-06-07 -> 2026-07-02)
Fold 2: train 1298 rows (2026-05-09 -> 2026-07-02) | test 600 rows (2026-07-02 -> 2026-07-27)
Fold 3: train 1898 rows (2026-05-09 -> 2026-07-27) | test 600 rows (2026-07-27 -> 2026-08-21)
Fold 4: train 2498 rows (2026-05-09 -> 2026-08-21) | test 600 rows (2026-08-21 -> 2026-09-15)
Fold 5: train 3098 rows (2026-05-09 -> 2026-09-15) | test 600 rows (2026-09-15 -> 2026-10-10)
Fold 6: train 3698 rows (2026-05-09 -> 2026-10-10) | test 600 rows (2026-10-10 -> 2026-11-04)


In [35]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

rows = []
for i, (tr_idx, te_idx) in enumerate(tscv.split(df)):
    tr, te = df.iloc[tr_idx], df.iloc[te_idx]
    Xtr, ytr = tr[feature_cols], tr["next_up"]
    Xte, yte = te[feature_cols], te["next_up"]

    # Baselines
    majority = int(ytr.mean() > 0.5)
    base_major = accuracy(pd.Series(majority, index=te.index), yte)
    base_opp = accuracy(1 - (te["return"] > 0).astype(int), yte)

    # Models, retrained from scratch on each fold's training data
    lr = make_pipeline(StandardScaler(), LogisticRegression()).fit(Xtr, ytr)
    xg = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                       subsample=0.8, colsample_bytree=0.8,
                       random_state=42).fit(Xtr, ytr)

    rows.append({
        "fold": i + 1,
        "test_period": f"{te['time'].iloc[0].date()} -> {te['time'].iloc[-1].date()}",
        "always_majority": base_major,
        "opposite_last": base_opp,
        "logreg": accuracy(lr.predict(Xte), yte),
        "xgboost": accuracy(xg.predict(Xte), yte),
    })

wf = pd.DataFrame(rows)
print(wf.to_string(index=False))
print()
print("Average across folds:")
print(wf[["always_majority", "opposite_last", "logreg", "xgboost"]].mean().round(1))

 fold              test_period  always_majority  opposite_last  logreg  xgboost
    1 2026-06-07 -> 2026-07-02             51.8           53.2    56.3     52.8
    2 2026-07-02 -> 2026-07-27             48.5           53.0    50.8     48.5
    3 2026-07-27 -> 2026-08-21             48.2           51.7    46.5     52.2
    4 2026-08-21 -> 2026-09-15             50.8           53.8    52.3     52.0
    5 2026-09-15 -> 2026-10-10             48.0           51.3    50.2     51.2
    6 2026-10-10 -> 2026-11-04             51.5           53.2    53.8     52.2

Average across folds:
always_majority    49.8
opposite_last      52.7
logreg             51.6
xgboost            51.5
dtype: float64


In [36]:
strategies = {}

def add(name, value):
    strategies.setdefault(name, []).append(value)

for i, (tr_idx, te_idx) in enumerate(tscv.split(df)):
    tr, te = df.iloc[tr_idx], df.iloc[te_idx]
    Xtr, ytr = tr[feature_cols], tr["next_up"]
    Xte = te[feature_cols]

    lr = make_pipeline(StandardScaler(), LogisticRegression()).fit(Xtr, ytr)
    xg = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                       subsample=0.8, colsample_bytree=0.8,
                       random_state=42).fit(Xtr, ytr)

    probs = {"LogReg": lr.predict_proba(Xte)[:, 1],
             "XGBoost": xg.predict_proba(Xte)[:, 1]}

    # The one-line rule: bet against the last hour's direction
    opp = 1 - (te["return"] > 0).astype(int)
    _, net, _ = backtest(opp, te)
    add("Opposite of last hour", (1 + net).prod() - 1)

    for name, p in probs.items():
        for thr in [0.50, 0.55, 0.60]:
            _, net, _ = backtest((p > thr).astype(int), te)
            add(f"{name} > {thr:.2f}", (1 + net).prod() - 1)

    add("Cash (do nothing)", 0.0)
    add("Buy and hold", (1 + te["next_ret"]).prod() - 1)

fold_cols = [f"F{i}" for i in range(1, 7)]
res = pd.DataFrame(strategies).T * 100
res.columns = fold_cols
res["folds_up"] = (res[fold_cols] > 0).sum(axis=1)
res["total_%"] = ((1 + res[fold_cols] / 100).prod(axis=1) - 1) * 100

print(res.round(1).to_string())

                         F1    F2    F3    F4    F5    F6  folds_up  total_%
Opposite of last hour -31.1 -25.1 -22.6 -26.5 -20.7 -30.1         0    -83.7
LogReg > 0.50           2.8  -5.5 -13.1 -14.6 -12.6 -15.4         1    -46.6
LogReg > 0.55          -0.5  -0.6  -4.6  -0.6  -5.2  -0.3         0    -11.4
LogReg > 0.60          -0.8  -0.5  -0.8  -0.1  -1.7  -1.4         0     -5.2
XGBoost > 0.50        -18.7 -19.9 -15.2 -23.0 -19.6 -19.9         0    -72.6
XGBoost > 0.55        -24.4 -13.7 -13.1 -12.1 -14.5 -11.1         0    -62.1
XGBoost > 0.60         -9.8  -7.8  -8.8  -4.3  -6.5  -4.3         0    -35.1
Cash (do nothing)       0.0   0.0   0.0   0.0   0.0   0.0         0      0.0
Buy and hold           -0.6   5.2  19.4  -1.4   8.7  -3.0         3     29.7
